## 서버 관련

In [3]:
import socket
import threading
import json
import time
import random


# =========================
# 서버 설정
# =========================

HOST = "10.10.59.205"
PORT = 5000

WIN_SCORE = 5


# =========================
# 게임 상태
# =========================

clients = {}
scores = {}

next_player_id = 1

current_round = 1
round_finished = False
game_finished = False

winner_player_id = -1
answer_index = -1

game_lock = threading.Lock()

answer_index = random.randint(0, 2)


# =========================
# 메시지 전송
# =========================

def send_message(client_socket, data):
    message = json.dumps(data) + "\n"

    client_socket.sendall(
        message.encode("utf-8")
    )


def broadcast(data):
    disconnected_players = []

    for player_id, client_socket in list(clients.items()):
        try:
            send_message(
                client_socket,
                data
            )

        except Exception as e:
            print(
                f"Player {player_id} 전송 실패:",
                e
            )

            disconnected_players.append(
                player_id
            )

    for player_id in disconnected_players:
        remove_player(player_id)


# =========================
# 플레이어 제거
# =========================

def remove_player(player_id):
    client_socket = clients.pop(
        player_id,
        None
    )

    scores.pop(
        player_id,
        None
    )

    if client_socket is not None:
        try:
            client_socket.close()
        except:
            pass

    print(
        f"Player {player_id} 연결 종료"
    )


# =========================
# 현재 상태 전송
# =========================

def send_game_state():
    broadcast({
        "type": "game_state",

        "round": current_round,

        "scores": scores,

        "game_finished": game_finished,

        "winner": winner_player_id,

        "answer_index": answer_index
    })


# =========================
# 라운드 시작
# =========================

def start_round():
    global round_finished
    global answer_index

    round_finished = False

    answer_index = random.randint(0, 2)
    print()
    print(
        f"===== Round {current_round} 시작 ====="
    )

    broadcast({
        "type": "round_start",

        "round": current_round,

        "scores": scores,

        "answer_index": answer_index
    })


# =========================
# 이미지 인식 성공 처리
# =========================

def handle_detection(
    player_id,
    round_id
):
    global current_round
    global round_finished
    global game_finished
    global winner_player_id

    with game_lock:

        # 이미 게임 끝남
        if game_finished:
            return

        # 존재하지 않는 플레이어
        if player_id not in scores:
            return

        # 이전 / 잘못된 라운드 메시지
        if round_id != current_round:

            print(
                f"Player {player_id} 잘못된 라운드 요청 "
                f"(받음: {round_id}, 현재: {current_round})"
            )

            return


        # 이미 다른 사람이 먼저 성공함
        if round_finished:

            print(
                f"Player {player_id} 인식 성공했지만 늦음"
            )

            return


        # =========================
        # 이 플레이어가 최초 성공자
        # =========================

        round_finished = True

        scores[player_id] += 1

        print()
        print(
            f"Round {current_round} 승자: "
            f"Player {player_id}"
        )

        print(
            "현재 점수:",
            scores
        )


        # =========================
        # 최종 승리 판정
        # =========================

        if scores[player_id] >= WIN_SCORE:

            game_finished = True
            winner_player_id = player_id

            print()
            print(
                "============================"
            )

            print(
                f"게임 최종 승자: "
                f"Player {player_id}"
            )

            print(
                "============================"
            )

            broadcast({
                "type": "game_over",

                "winner": player_id,

                "round": current_round,

                "scores": scores
            })

            return


        # =========================
        # 라운드 결과 전송
        # =========================

        broadcast({
            "type": "round_result",

            "round": current_round,

            "winner": player_id,

            "scores": scores,

            "answer_index": answer_index
        })


    # Lock 밖에서 잠깐 대기
    # 결과 화면을 보여주기 위한 시간
    time.sleep(1.0)


    with game_lock:

        if game_finished:
            return

        current_round += 1


    start_round()


# =========================
# 클라이언트 메시지 처리
# =========================

def handle_message(
    player_id,
    message
):
    message_type = message.get(
        "type"
    )


    # 이미지 인식 성공
    if message_type == "detected":

        round_id = message.get(
            "round"
        )

        if round_id is None:
            return

        handle_detection(
            player_id,
            round_id
        )


    # 준비 상태 확인용
    elif message_type == "ready":

        print(
            f"Player {player_id} Ready"
        )


    # Ping 테스트
    elif message_type == "ping":

        client_socket = clients.get(
            player_id
        )

        if client_socket:

            send_message(
                client_socket,
                {
                    "type": "pong"
                }
            )


    else:

        print(
            f"알 수 없는 메시지 "
            f"Player {player_id}:",
            message
        )


# =========================
# 클라이언트 연결 처리
# =========================

def handle_client(
    client_socket,
    address,
    player_id
):

    print()
    print(
        f"Player {player_id} 접속"
    )

    print(
        "주소:",
        address
    )


    # 자신의 플레이어 번호 전달
    send_message(
        client_socket,
        {
            "type": "connected",

            "player_id": player_id,

            "round": current_round,

            "scores": scores,

            "answer_index": answer_index
        }
    )


    # 다른 플레이어에게 접속 알림
    broadcast({
        "type": "player_joined",

        "player_id": player_id,

        "scores": scores
    })


    buffer = ""


    try:

        while True:

            data = client_socket.recv(
                4096
            )


            # 연결 종료
            if not data:
                break


            buffer += data.decode(
                "utf-8"
            )


            # TCP에서는 메시지가 붙어서 올 수 있기 때문에
            # \n 기준으로 분리
            while "\n" in buffer:

                line, buffer = buffer.split(
                    "\n",
                    1
                )


                if not line.strip():
                    continue


                try:

                    message = json.loads(
                        line
                    )

                except json.JSONDecodeError:

                    print(
                        f"Player {player_id} "
                        f"JSON 오류:",
                        line
                    )

                    continue


                handle_message(
                    player_id,
                    message
                )


    except ConnectionResetError:

        print(
            f"Player {player_id} "
            "연결 강제 종료"
        )


    except Exception as e:

        print(
            f"Player {player_id} 오류:",
            e
        )


    finally:

        remove_player(
            player_id
        )

        broadcast({
            "type": "player_left",

            "player_id": player_id,

            "scores": scores
        })


# =========================
# 서버 생성
# =========================

server_socket = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)


# 서버 재실행 시
# Address already in use 방지
server_socket.setsockopt(
    socket.SOL_SOCKET,
    socket.SO_REUSEADDR,
    1
)


server_socket.bind(
    (
        HOST,
        PORT
    )
)


server_socket.listen()


print(
    "============================"
)

print(
    "게임 서버 시작"
)

print(
    f"IP   : {HOST}"
)

print(
    f"PORT : {PORT}"
)

print(
    f"승리 조건 : {WIN_SCORE}점"
)

print(
    "============================"
)


# =========================
# 접속 대기
# =========================

try:

    while True:

        client_socket, address = (
            server_socket.accept()
        )


        with game_lock:

            player_id = next_player_id
            next_player_id += 1

            clients[player_id] = (
                client_socket
            )

            scores[player_id] = 0


        thread = threading.Thread(
            target=handle_client,

            args=(
                client_socket,
                address,
                player_id
            ),

            daemon=True
        )

        thread.start()


except KeyboardInterrupt:

    print()
    print(
        "서버 종료"
    )


finally:

    for client_socket in clients.values():

        try:
            client_socket.close()

        except:
            pass


    server_socket.close()

게임 서버 시작
IP   : 10.10.59.205
PORT : 5000
승리 조건 : 5점

Player 1 접속
주소: ('10.10.59.206', 52346)

Player 2 접속
주소: ('10.10.59.205', 46090)

Round 1 승자: Player 2
현재 점수: {1: 0, 2: 1}

===== Round 2 시작 =====

Round 2 승자: Player 2
현재 점수: {1: 0, 2: 2}

===== Round 3 시작 =====

Round 3 승자: Player 1
현재 점수: {1: 1, 2: 2}

===== Round 4 시작 =====

Round 4 승자: Player 1
현재 점수: {1: 2, 2: 2}

===== Round 5 시작 =====

Round 5 승자: Player 1
현재 점수: {1: 3, 2: 2}

===== Round 6 시작 =====

Round 6 승자: Player 1
현재 점수: {1: 4, 2: 2}

===== Round 7 시작 =====
Player 1 연결 종료
Player 2 연결 종료

서버 종료
